In [1]:
# ============================================================
# CONFIGURAÇÃO INICIAL DO AMBIENTE
# ============================================================
#
# Este notebook constrói a camada Bronze da pipeline.
#
# Nesta célula, configuramos:
#
#   - raiz do projeto;
#   - diretório local dos arquivos extraídos;
#   - interpretador Python usado pelo PySpark;
#   - credenciais temporárias da AWS;
#   - bucket S3 do integrante.
#
# Cada integrante deve executar o notebook com seu próprio arquivo .env,
# suas próprias credenciais do AWS Academy Learner Lab e seu próprio
# bucket S3.
#
# Pontos de configuração esperados no .env:
#
#   AWS_ACCESS_KEY_ID      Chave de acesso temporária da AWS.
#   AWS_SECRET_ACCESS_KEY  Chave secreta temporária da AWS.
#   AWS_SESSION_TOKEN      Token temporário da sessão AWS Academy.
#   S3_BUCKET              Nome do bucket S3 do integrante.
#
# ============================================================
# BIBLIOTECAS
# ============================================================

# Setup Env: Dotenv/Pathlib
import os, sys
from pathlib import Path
from dotenv import load_dotenv, find_dotenv

# ============================================================
# CAMINHOS DO PROJETO E VARIÁVEIS DE AMBIENTE
# ============================================================

# Localiza o arquivo .env, carrega suas variáveis e define a raiz do projeto.
# O diretório EXTRAIDOS_DIR contém os arquivos gerados no notebook
# 00_aquisicao_dados.ipynb.
load_dotenv(find_dotenv())
PROJECT_ROOT = Path(find_dotenv()).parent
EXTRAIDOS_DIR = PROJECT_ROOT / "data_lake" / "external" / "extraidos"

# ============================================================
# CONFIGURAÇÃO DO PYSPARK
# ============================================================

# Garante que o PySpark use o mesmo interpretador Python do ambiente atual.
# Isso evita conflitos comuns em WSL, Conda ou ambientes virtuais.
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

# ============================================================
# CREDENCIAIS AWS E BUCKET S3
# ============================================================

# Carrega as credenciais temporárias e o bucket S3 definidos no .env.
AWS_ACCESS_KEY_ID     = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")
AWS_SESSION_TOKEN     = os.getenv("AWS_SESSION_TOKEN")
S3_BUCKET             = os.getenv("S3_BUCKET")

# Interrompe a execução se alguma configuração obrigatória não foi carregada.
if not all([AWS_ACCESS_KEY_ID, AWS_SECRET_ACCESS_KEY, AWS_SESSION_TOKEN, S3_BUCKET]):
    raise EnvironmentError("Falha ao carregar credenciais AWS ou S3_BUCKET do arquivo .env")

# ============================================================
# CONFERÊNCIA DA CONFIGURAÇÃO
# ============================================================

print(f"Projeto: {PROJECT_ROOT}\nBucket : {S3_BUCKET}")

Projeto: /mnt/d/diego/01_projects/postech-challenge-2
Bucket : alfabetizacao-data-lake-diego


In [2]:
# ============================================================
# CRIAÇÃO DA SESSÃO SPARK COM ACESSO AO S3
# ============================================================
#
# Esta célula cria a SparkSession usada para processar os arquivos
# locais extraídos e gravar os resultados da camada Bronze no S3.
#
# A sessão é executada localmente com master("local[*]"), usando os
# núcleos disponíveis da máquina.
#
# Pacotes configurados:
#
#   hadoop-aws
#       Permite acesso ao S3 via protocolo s3a://.
#
#   aws-java-sdk-bundle
#       Fornece dependências da AWS usadas pelo conector S3A.
#
#   spark-excel
#       Permite leitura de arquivos XLSX com Spark.
#
# Como o AWS Academy Learner Lab usa credenciais temporárias, a sessão
# Spark utiliza TemporaryAWSCredentialsProvider e recebe também o
# AWS_SESSION_TOKEN.
#
# Entrada:
#
#   AWS_ACCESS_KEY_ID
#   AWS_SECRET_ACCESS_KEY
#   AWS_SESSION_TOKEN
#
# Saída:
#
#   spark  Sessão Spark configurada.
#
# ============================================================
# BIBLIOTECAS
# ============================================================

# SparkSession: S3 Config
from pyspark.sql import SparkSession

# ============================================================
# SESSÃO SPARK
# ============================================================

spark = (
    SparkSession.builder
    .appName("exploracao-e-armazenar-S3")
    .master("local[*]")
    # Configuração de Pacotes e S3A
    .config("spark.jars.packages","org.apache.hadoop:hadoop-aws:3.3.4,""com.amazonaws:aws-java-sdk-bundle:1.12.262,""com.crealytics:spark-excel_2.12:3.5.1_0.20.4")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.TemporaryAWSCredentialsProvider")
    .config("spark.hadoop.fs.s3a.access.key", AWS_ACCESS_KEY_ID)
    .config("spark.hadoop.fs.s3a.secret.key", AWS_SECRET_ACCESS_KEY)
    .config("spark.hadoop.fs.s3a.session.token", AWS_SESSION_TOKEN)
    .config("spark.hadoop.fs.s3a.endpoint", "s3.amazonaws.com")
    .getOrCreate()
)

# ============================================================
# CONFERÊNCIA DA SESSÃO
# ============================================================

print(f"Sessão Spark {spark.version} criada com sucesso.")

26/07/07 00:27:20 WARN Utils: Your hostname, lua resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/07/07 00:27:20 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/diego/miniconda3/envs/postech2/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/diego/.ivy2/cache
The jars for the packages stored in: /home/diego/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
com.crealytics#spark-excel_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-85e53e82-1896-4c5a-aac5-b9e48d79a151;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
	found com.crealytics#spark-excel_2.12;3.5.1_0.20.4 in central
	found org.apache.poi#poi;5.2.5 in central
	found org.apache.commons#commons-collections4;4.4 in central
	found org.apache.commons#commons-math3;3.6.1 in central
	found com.zaxxer#SparseBitSet;1.3 in central
	found org.apache.poi#poi-ooxml;5.2.5 in central
	found org.apache.poi#poi-ooxml-lite;5.2.5 in central
	found com.github.virtuald#curvesapi;1.08 in centr

Sessão Spark 3.5.5 criada com sucesso.


In [3]:
# ============================================================
# CLIENTE S3 PARA OPERAÇÕES ADMINISTRATIVAS
# ============================================================
#
# Esta célula cria um cliente S3 com boto3 usando as mesmas
# credenciais temporárias carregadas do arquivo .env.
#
# O Spark será usado para processar e gravar os dados da camada
# Bronze. O boto3 fica reservado para operações administrativas
# no bucket, como listar, validar ou limpar prefixos quando necessário.
#
# Entrada:
#
#   AWS_ACCESS_KEY_ID
#   AWS_SECRET_ACCESS_KEY
#   AWS_SESSION_TOKEN
#
# Saída:
#
#   s3_client  Cliente boto3 configurado para acessar o S3.
#
# ============================================================
# BIBLIOTECAS
# ============================================================

# Boto3: Gestão de pastas
import boto3

# ============================================================
# CLIENTE S3
# ============================================================

# Inicializa o cliente S3 logo após o setup das credenciais.
s3_client = boto3.client(
    's3',
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    aws_session_token=AWS_SESSION_TOKEN
)

## Landing

Nesta etapa, copiamos para o S3 os arquivos crus que foram baixados e extraídos localmente.

A camada Landing funciona como uma área de entrada no bucket: os arquivos são enviados praticamente como estão, preservando formato, nomes e organização por ano. Ainda não há transformação de dados nesta etapa.

O objetivo é garantir que cada integrante consiga subir seus próprios arquivos para seu próprio bucket, usando as credenciais configuradas no `.env`.

In [5]:
# ====================================================================
# LANDING — ARQUIVOS CRUS NO S3
# ====================================================================
#
# Esta célula envia para a camada Landing do S3 os arquivos locais
# presentes em data_lake/external/extraidos.
#
# A Landing preserva os arquivos em seu formato original, como CSV
# e XLSX, sem aplicar transformação de dados.
#
# Estrutura esperada no S3:
#
#   s3://bucket-do-integrante/landing/2023/...
#   s3://bucket-do-integrante/landing/2024/...
#   s3://bucket-do-integrante/landing/2025/...
#
# Entrada:
#
#   EXTRAIDOS_DIR  Diretório local com arquivos extraídos.
#   S3_BUCKET      Bucket S3 do integrante.
#   s3_client      Cliente boto3 configurado.
#
# Saída:
#
#   Arquivos enviados para o prefixo landing/ no S3.
#
# ====================================================================
# BIBLIOTECAS
# ====================================================================

from botocore.exceptions import ClientError

# ====================================================================
# CONFIGURAÇÕES DA LANDING
# ====================================================================

LANDING_PREFIX    = "landing"
EXTENSOES_LANDING = {".csv", ".xlsx"}

# ====================================================================
# LISTAGEM DE ARQUIVOS EXISTENTES NO S3
# ====================================================================

def keys_no_landing():
    """Set de keys já no Landing. Retorna None se a credencial não puder listar."""
    try:
        paginator = s3_client.get_paginator("list_objects_v2")
        return {o["Key"]
                for page in paginator.paginate(Bucket=S3_BUCKET, Prefix=f"{LANDING_PREFIX}/")
                for o in page.get("Contents", [])}
    except ClientError as e:
        if e.response["Error"]["Code"] in {"AccessDenied", "403"}:
            print("Aviso: sem permissão de listar (s3:ListBucket) — subindo tudo sem checar existência.\n")
            return None
        raise   # erro real (rede, credencial expirada) -> falha alto

# ====================================================================
# UPLOAD DOS ARQUIVOS PARA A LANDING
# ====================================================================

def subir_landing(force: bool = False):
    # existentes=None -> ou force, ou sem permissão de listar -> sobe tudo
    existentes = None if force else keys_no_landing()
    modo = ("reenvio forçado" if force else
            "checando existência" if existentes is not None else
            "upload incondicional")
    print(f"Modo: {modo}\n")

    arquivos = sorted(
        arq for arq in EXTRAIDOS_DIR.rglob("*")
        if arq.is_file() and arq.suffix.lower() in EXTENSOES_LANDING
    )
    enviados = pulados = 0
    for arq in arquivos:
        rel = arq.relative_to(EXTRAIDOS_DIR).as_posix()
        key = f"{LANDING_PREFIX}/{rel}"
        if existentes is not None and key in existentes:
            pulados += 1
            continue
        s3_client.upload_file(str(arq), S3_BUCKET, key)
        enviados += 1
        print(f"  ↑ {key}  ({arq.stat().st_size / 1e6:.1f} MB)")

    print(f"\nLanding: {enviados} enviados | {pulados} já existiam | {len(arquivos)} locais.")

# ====================================================================
# EXECUÇÃO
# ====================================================================

subir_landing()

Modo: checando existência


Landing: 0 enviados | 15 já existiam | 15 locais.


## Bronze: CSV

Nesta etapa, os arquivos CSV extraídos localmente são lidos com Spark, tipados de acordo com contratos explícitos e gravados na camada Bronze do S3 em formato Parquet.

A Bronze mantém o dado próximo ao original, mas já aplica padronizações técnicas necessárias para a pipeline, como tipos de dados, metadados de rastreabilidade e particionamento por ano de avaliação.

In [9]:
from datetime import datetime, timezone
from pyspark.sql import functions as F

# Bases parametrizadas
INPUT_BASE  = str(EXTRAIDOS_DIR)            # Desenvolvimento: arquivos locais
BRONZE_BASE = f"s3a://{S3_BUCKET}/bronze"   # Produção: Bronze no S3
ANOS = ["2023", "2024", "2025"]

# Timestamp único por execução (determinístico para todo o batch)
INGESTION_TS = datetime.now(timezone.utc)

# Sobrescreve apenas as partições afetadas
spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")

CSV_OPTS = {
    "header": "true",
    "sep": ";",
    "encoding": "ISO-8859-1",
}

# --------------------------------------------------------------------
# Contratos de tipo (codigo=string ; medida=double ; flag=int)
# --------------------------------------------------------------------
SCHEMA_TS_ALUNO = [
    ("NU_ANO_AVALIACAO", "int"),
    ("CO_UF", "string"),
    ("SG_UF", "string"),
    ("ID_ALUNO", "string"),
    ("TP_SERIE", "string"),
    ("ID_ESCOLA", "string"),
    ("TP_DEPENDENCIA", "string"),
    ("CO_MUNICIPIO", "string"),
    ("NO_MUNICIPIO", "string"),
    ("IN_PRESENCA_LP", "int"),
    ("IN_PREENCHIMENTO_LP", "int"),
    ("CO_CADERNO_LP", "string"),
    ("VL_PESO_ALUNO_LP", "double"),
    ("VL_PROFICIENCIA_LP", "double"),
    ("IN_ALFABETIZADO", "int"),
]
SCHEMA_TS_MUNICIPIO = [
    ("NU_ANO_AVALIACAO", "int"),
    ("CO_UF", "string"),
    ("SG_UF", "string"),
    ("CO_MUNICIPIO", "string"),
    ("NO_MUNICIPIO", "string"),
    ("TP_SERIE", "string"),
    ("ID_TIPO_REDE", "string"),
    ("PC_ALUNO_ALFABETIZADO", "double"),
    ("VL_MEDIA_LP", "double"),
]
SCHEMA_TS_ESTADO = [
    ("NU_ANO_AVALIACAO", "int"),
    ("CO_UF", "string"),
    ("SG_UF", "string"),
    ("TP_SERIE", "string"),
    ("ID_TIPO_REDE", "string"),
    ("PC_ALUNO_ALFABETIZADO", "double"),
    ("VL_MEDIA_LP", "double"),
]

# --------------------------------------------------------------------
# Ingestão Bronze genérica
# --------------------------------------------------------------------
def bronze_csv(entidade: str, arquivo: str, schema: list):
    # Multicaminho: 1 scan, Spark paraleliza a leitura. Tudo string.
    caminhos = [f"{INPUT_BASE}/{ano}/{arquivo}" for ano in ANOS]
    raw = spark.read.options(**CSV_OPTS).csv(caminhos)

    # Cast por NOME (order-safe; coluna ausente estoura aqui).
    # Vazio/'-' -> null no cast (parsing, não regra de negócio).
    dados = [F.col(nome).cast(tipo).alias(nome) for nome, tipo in schema]
    df = raw.select(
        *dados,
        F.input_file_name().alias("_source_file"),          # lineage
        F.lit(INGESTION_TS).alias("_ingestion_timestamp"),  # carimbo único do batch
    )

    # Particiona pela coluna do próprio dado (confiamos que está preenchida).
    destino = f"{BRONZE_BASE}/{entidade}"
    (df.write
        .mode("overwrite")
        .partitionBy("NU_ANO_AVALIACAO")
        .parquet(destino))
    return df

In [7]:
# ============================================================
# FUNÇÃO GENÉRICA PARA INGESTÃO BRONZE DE CSV
# ============================================================
#
# Esta célula define a função responsável por ingerir arquivos CSV
# para a camada Bronze.
#
# A função lê os arquivos ano a ano, valida se as colunas esperadas
# existem pelo nome, aplica os tipos definidos no contrato, adiciona
# metadados de rastreabilidade e grava o resultado em Parquet no S3.
#
# Essa abordagem é importante porque os arquivos podem mudar entre
# anos. Por exemplo, em 2025 o arquivo TS_ALUNO.csv passou a conter
# novas colunas. Como a função seleciona os campos pelo nome, novas
# colunas fora do contrato são ignoradas, e mudanças de posição não
# corrompem os dados.
#
# Se uma coluna esperada mudar de nome ou desaparecer, a execução
# falha com mensagem clara, evitando erro silencioso na pipeline.
#
# Entrada:
#
#   entidade  Nome da entidade no destino Bronze.
#   arquivo   Nome do CSV dentro de cada pasta anual.
#   schema    Lista com pares de nome da coluna e tipo esperado.
#
# Saída:
#
#   DataFrame Spark tipado e gravado em:
#   s3a://{S3_BUCKET}/bronze/{entidade}/
#
# ============================================================
# BIBLIOTECAS
# ============================================================

from functools import reduce

# ============================================================
# FUNÇÃO DE INGESTÃO
# ============================================================

def bronze_csv(entidade: str, arquivo: str, schema: list):
    # Extrai do contrato apenas os nomes das colunas esperadas.
    nomes  = [n for n, _ in schema]

    # Lista usada para armazenar o DataFrame de cada ano antes da união.
    partes = []

    for ano in ANOS:
        # Lê um ano por vez para que cada arquivo use seu próprio cabeçalho.
        # Isso protege contra mudanças na posição das colunas entre os anos.
        raw = spark.read.options(**CSV_OPTS).csv(f"{INPUT_BASE}/{ano}/{arquivo}")

        # Valida se todas as colunas do contrato existem no arquivo lido.
        # Se alguma coluna esperada estiver ausente, a execução é interrompida.
        faltando = [n for n in nomes if n not in raw.columns]
        assert not faltando, f"{arquivo} ({ano}) sem colunas do contrato: {faltando}"

        # Seleciona as colunas pelo nome e aplica os tipos definidos no contrato.
        dados = [F.col(n).cast(t).alias(n) for n, t in schema]

        # Adiciona metadados técnicos para rastrear a origem e o momento da ingestão.
        partes.append(raw.select(
            *dados,
            F.input_file_name().alias("_source_file"),
            F.lit(INGESTION_TS).alias("_ingestion_timestamp"),
        ))

    # Une os DataFrames anuais pelo nome das colunas.
    df = reduce(lambda a, b: a.unionByName(b), partes)

    # Grava a entidade na Bronze em formato Parquet, particionada por ano.
    (df.write.mode("overwrite").partitionBy("NU_ANO_AVALIACAO")
        .parquet(f"{BRONZE_BASE}/{entidade}"))

    return df

In [ ]:
# ============================================================
# EXECUÇÃO DA INGESTÃO BRONZE DOS CSVS
# ============================================================
#
# Esta célula executa a função bronze_csv para as três entidades
# principais da pipeline:
#
#   - ts_aluno;
#   - ts_municipio;
#   - ts_estado.
#
# Para cada entidade, a função lê os arquivos dos anos configurados
# em ANOS, aplica o contrato de schema correspondente e grava o
# resultado na camada Bronze do S3.
#
# Saídas no S3:
#
#   s3a://{S3_BUCKET}/bronze/ts_aluno/
#   s3a://{S3_BUCKET}/bronze/ts_municipio/
#   s3a://{S3_BUCKET}/bronze/ts_estado/
#
# ============================================================
# EXECUÇÃO
# ============================================================

#df_aluno     = bronze_csv("ts_aluno",     "TS_ALUNO.csv",     SCHEMA_TS_ALUNO)
#df_municipio = bronze_csv("ts_municipio", "TS_MUNICIPIO.csv", SCHEMA_TS_MUNICIPIO)
#df_estado    = bronze_csv("ts_estado",    "TS_ESTADO.csv",    SCHEMA_TS_ESTADO)

26/07/06 18:10:33 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
26/07/06 18:10:43 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/07/06 18:10:43 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/07/06 18:10:43 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/07/06 18:10:43 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/07/06 18:10:43 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/07/06 18:10:43 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,05

In [9]:
# ============================================================
# VALIDAÇÃO DA CAMADA BRONZE DOS CSVS
# ============================================================
#
# Esta célula lê de volta os dados gravados no S3 para validar se
# a ingestão Bronze foi concluída corretamente.
#
# As validações feitas são:
#
#   - contagem de linhas por entidade;
#   - conferência das partições esperadas;
#   - inspeção do schema gravado em Parquet;
#   - contagem de registros por ano;
#   - verificação das colunas de rastreabilidade;
#   - contagem de proficiência nula em TS_ALUNO.
#
# A leitura é feita a partir do S3 para validar o dado persistido,
# e não apenas os DataFrames que ficaram em memória.
#
# ============================================================
# VALIDAÇÃO DE PARTIÇÕES E VOLUME
# ============================================================

# Count + guard-rail das partições, lendo de volta do S3 (sem recomputar o pipeline).
# distinct sobre a coluna de partição é barato (vem dos diretórios, não varre os dados).
for entidade in ["ts_aluno", "ts_municipio", "ts_estado"]:
    dfc  = spark.read.parquet(f"{BRONZE_BASE}/{entidade}")
    n    = dfc.count()
    anos = sorted(r["NU_ANO_AVALIACAO"] for r in
                  dfc.select("NU_ANO_AVALIACAO").distinct().collect())
    assert anos == [2023, 2024, 2025], f"{entidade}: anos inesperados na partição -> {anos}"
    print(f"{entidade}: {n:,} linhas | partições {anos}")

# ============================================================
# VALIDAÇÃO DE SCHEMA E CONTAGEM POR ANO
# ============================================================

# Schema tipado + contagem por ano vindos de volta do Parquet
chk = spark.read.parquet(f"{BRONZE_BASE}/ts_municipio")
chk.printSchema()
chk.groupBy("NU_ANO_AVALIACAO").count().orderBy("NU_ANO_AVALIACAO").show()

# ============================================================
# VALIDAÇÃO DAS COLUNAS DE RASTREABILIDADE
# ============================================================

# Colunas de controle chegaram?
chk.select("_source_file", "_ingestion_timestamp").show(3, truncate=False)

# ============================================================
# SANITY CHECK DE PROFICIÊNCIA NULA
# ============================================================

# Sanity do cast: ausentes viram null na proficiência (não '-', não 0)
n_nulos = (spark.read.parquet(f"{BRONZE_BASE}/ts_aluno")
                .filter(F.col("VL_PROFICIENCIA_LP").isNull())
                .count())
print(f"TS_ALUNO com proficiência nula: {n_nulos:,}")

ts_aluno: 6,090,791 linhas | partições [2023, 2024, 2025]


ts_municipio: 36,411 linhas | partições [2023, 2024, 2025]


ts_estado: 225 linhas | partições [2023, 2024, 2025]
root
 |-- CO_UF: string (nullable = true)
 |-- SG_UF: string (nullable = true)
 |-- CO_MUNICIPIO: string (nullable = true)
 |-- NO_MUNICIPIO: string (nullable = true)
 |-- TP_SERIE: string (nullable = true)
 |-- ID_TIPO_REDE: string (nullable = true)
 |-- PC_ALUNO_ALFABETIZADO: double (nullable = true)
 |-- VL_MEDIA_LP: double (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- NU_ANO_AVALIACAO: integer (nullable = true)



+----------------+-----+
|NU_ANO_AVALIACAO|count|
+----------------+-----+
|            2023|11547|
|            2024|12448|
|            2025|12416|
+----------------+-----+



+------------------------------------------------------------------------------------------------------+--------------------------+
|_source_file                                                                                          |_ingestion_timestamp      |
+------------------------------------------------------------------------------------------------------+--------------------------+
|file:///mnt/d/diego/01_projects/postech-challenge-2/data_lake/external/extraidos/2023/TS_MUNICIPIO.csv|2026-07-06 18:10:20.605832|
|file:///mnt/d/diego/01_projects/postech-challenge-2/data_lake/external/extraidos/2023/TS_MUNICIPIO.csv|2026-07-06 18:10:20.605832|
|file:///mnt/d/diego/01_projects/postech-challenge-2/data_lake/external/extraidos/2023/TS_MUNICIPIO.csv|2026-07-06 18:10:20.605832|
+------------------------------------------------------------------------------------------------------+--------------------------+
only showing top 3 rows



TS_ALUNO com proficiência nula: 769,525


In [10]:
# ============================================================
# SANITY CHECK: PROFICIÊNCIA NULA E ALUNOS AUSENTES
# ============================================================
#
# Esta célula verifica se a quantidade de registros com proficiência
# nula em Língua Portuguesa corresponde à quantidade de alunos ausentes.
#
# A checagem ajuda a confirmar uma premissa importante para as próximas
# camadas: alunos ausentes não possuem proficiência calculada.
#
# Esta validação ainda não altera os dados. Ela apenas confirma se o
# comportamento observado na Bronze está coerente com a regra esperada.
#
# Entrada:
#
#   s3a://{S3_BUCKET}/bronze/ts_aluno/
#
# Saída:
#
#   Contagem de proficiências nulas.
#   Contagem de alunos ausentes.
#   Resultado booleano indicando se as contagens batem.
#
# ============================================================
# LEITURA DA BRONZE
# ============================================================

aluno     = spark.read.parquet(f"{BRONZE_BASE}/ts_aluno")

# ============================================================
# CÁLCULO DAS CONTAGENS
# ============================================================

prof_nula = aluno.filter(F.col("VL_PROFICIENCIA_LP").isNull()).count()
ausentes  = aluno.filter(F.col("IN_PRESENCA_LP") == 0).count()

# ============================================================
# RESULTADO DA CHECAGEM
# ============================================================

print(f"proficiência nula : {prof_nula:,}")
print(f"ausentes (=0)     : {ausentes:,}")
print("batem:", prof_nula == ausentes)

proficiência nula : 769,525
ausentes (=0)     : 765,024
batem: False


## Bronze: XLSX

Nesta etapa, ingerimos para a camada Bronze as planilhas XLSX de metas de 2023, 2024 e 2025.

Diferente dos CSVs principais, as planilhas de metas podem mudar bastante entre anos: nomes de colunas, formatos, sentinelas como `-` e valores textuais como `>80`. Por isso, os contratos são definidos por arquivo e por ano.

Na Bronze, preservamos esses detalhes técnicos sempre que necessário, deixando normalizações de regra de negócio para a camada Silver.

In [4]:
# ============================================================
# BRONZE: CONFIGURAÇÃO DA INGESTÃO DAS PLANILHAS DE METAS
# ============================================================
#
# Esta célula define os componentes utilizados durante a ingestão
# das planilhas XLSX de metas para a camada Bronze.
#
# São configurados:
#   - o leitor Spark para arquivos Excel;
#   - os contratos (schemas) utilizados na leitura;
#   - uma função auxiliar para reduzir repetição na definição
#     das colunas de metas.
#
# As planilhas de metas são tratadas separadamente dos arquivos CSV,
# pois apresentam estrutura própria e diferenças de schema entre
# os anos de publicação.

# ============================================================
# CONFIGURAÇÃO DO LEITOR EXCEL
# ============================================================
#
# Biblioteca Spark utilizada para leitura das planilhas XLSX.

SPARK_EXCEL = "com.crealytics.spark.excel"

# ============================================================
# CONTRATOS DAS PLANILHAS DE METAS
# ============================================================
#
# Cada contrato representa o schema esperado para uma combinação
# de entidade (Município ou UF) e ano.
#
# Os contratos são separados porque as planilhas de metas
# apresentam drift de schema entre 2023, 2024 e 2025.

# O contrato possui o ano no nome porque é válido apenas para
# as planilhas publicadas em 2023. Os schemas mudam entre anos,
# portanto cada versão possui seu próprio contrato.
SCHEMA_METAS_MUN_2023 = [
    ("ANO", "int"),
    ("CO_UF", "string"), ("SG_UF", "string"),
    ("CO_MUNICIPIO", "string"), ("NO_MUNICIPIO", "string"),
    ("NO_TP_REDE", "string"),
    ("PC_ALUNO_ALFABETIZADO", "double"),
    ("META_FINAL_2024", "double"), ("META_FINAL_2025", "double"),
    ("META_FINAL_2026", "double"), ("META_FINAL_2027", "double"),
    ("META_FINAL_2028", "double"), ("META_FINAL_2029", "double"),
    ("META_FINAL_2030", "double"),
    # Valores "-" são convertidos para null durante o cast.
    # Valores de 0 a 5 representam níveis válidos.
    ("NIVEIS_ALFABETIZACAO_2023", "int"),
    ("PC_AVALIADOS_LP", "double"),
]

SCHEMA_METAS_UF_2023 = [
    ("ANO", "int"),
    ("CD_UF", "string"),                       # PRESERVADO no Bronze (Silver -> CO_UF)
    ("SIGLA_UF", "string"), ("NOME_UF", "string"),
    ("REDE", "string"),
    ("SAEB_2019", "double"), ("SAEB_2021", "double"),   # numéricos limpos
    ("PC_ALUNO_ALFABETIZADO", "double"),                # só '- ' (suprimido) -> null
    # As metas permanecem como string na Bronze porque algumas
    # planilhas utilizam valores como ">80" ou "> 80".
    #
    # A conversão para valor numérico será realizada na Silver,
    # preservando a informação original durante a ingestão.
    ("META_FINAL_2024", "string"), ("META_FINAL_2025", "string"),
    ("META_FINAL_2026", "string"), ("META_FINAL_2027", "string"),
    ("META_FINAL_2028", "string"), ("META_FINAL_2029", "string"),
    ("META_FINAL_2030", "string"),
    ("PC_AVALIADOS_LP", "double"),                      # só '- ' (suprimido) -> null
]

def metas(tipo):   # gera META_FINAL_2024..2030 do mesmo tipo
    """
    Gera automaticamente as colunas META_FINAL_2024 até
    META_FINAL_2030 utilizando o tipo informado.

    Isso evita repetição na definição dos contratos.
    """
    return [(f"META_FINAL_{a}", tipo) for a in range(2024, 2031)]

# ------------------------------------------------------------
# MUNICÍPIOS - PLANILHAS 2024
#
# As metas são numéricas e podem ser lidas diretamente como
# double. O schema já utiliza a nomenclatura presente nas
# planilhas publicadas em 2024.
# ------------------------------------------------------------
SCHEMA_METAS_MUN_2024 = [
    ("ANO","int"), ("CO_UF","string"), ("SG_UF","string"),
    ("CO_MUNICIPIO","string"), ("NO_MUNICIPIO","string"), ("NO_TP_REDE","string"),
    ("PC_ALUNO_ALFABETIZADO_2023","double"), ("PC_ALUNO_ALFABETIZADO_2024","double"),
    *metas("double"),
    ("CO_NIVEL_ALFABETIZACAO","int"),
    ("PC_AVALIADOS_LP","double"),
]
SCHEMA_METAS_MUN_2025 = [
    ("ANO","int"), ("CO_UF","string"), ("SG_UF","string"),
    ("CO_MUNICIPIO","string"), ("NO_MUNICIPIO","string"), ("NO_TP_REDE","string"),
    ("PC_ALUNO_ALFABETIZADO_2023","double"), ("PC_ALUNO_ALFABETIZADO_2024","double"),
    ("PC_ALUNO_ALFABETIZADO_2025","double"),
    *metas("double"),
    ("CO_NIVEL_ALFABETIZACAO","int"),
    ("PC_AVALIADOS_LP","double"),
]

# ------------------------------------------------------------
# UF - PLANILHAS 2024
#
# As metas permanecem como string para preservar valores como
# ">80". A normalização será realizada posteriormente na
# camada Silver.
# ------------------------------------------------------------
SCHEMA_METAS_UF_2024 = [
    ("ANO","int"), ("CD_UF","string"), ("SIGLA_UF","string"), ("NOME_UF","string"),
    ("REDE","string"),
    ("PC_ALUNO_ALFABETIZADO_2023","double"), ("PC_ALUNO_ALFABETIZADO_2024","double"),
    *metas("string"),
    ("PC_AVALIADOS_LP","double"),
]
SCHEMA_METAS_UF_2025 = [
    ("ANO","int"), ("CD_UF","string"), ("SIGLA_UF","string"), ("NOME_UF","string"),
    ("REDE","string"),
    ("PC_ALUNO_ALFABETIZADO_2023","double"), ("PC_ALUNO_ALFABETIZADO_2024","double"),
    ("PC_ALUNO_ALFABETIZADO_2025","double"),
    *metas("string"),
    ("PC_AVALIADOS_LP","double"),
]

In [10]:
# ============================================================
# FUNÇÃO DE INGESTÃO DAS PLANILHAS XLSX
# ============================================================
#
# Realiza a ingestão de uma planilha XLSX para a camada Bronze.
#
# Etapas executadas:
#   1. Lê a planilha utilizando o conector Spark Excel.
#   2. Valida se todas as colunas previstas no contrato existem.
#   3. Aplica o schema definido para a combinação
#      (entidade, ano).
#   4. Adiciona metadados de rastreabilidade.
#   5. Remove registros que não representam dados válidos.
#   6. Grava o resultado em formato Parquet, particionado por ano.

def bronze_xlsx(entidade, arquivo, sheet, schema, ano):

    # Caminho da planilha de origem no Data Lake.
    caminho = f"{INPUT_BASE}/{ano}/{arquivo}"

    # Lê a planilha preservando todos os valores como texto.
    # A conversão de tipos será realizada utilizando o contrato.
    raw = (
        spark.read.format(SPARK_EXCEL)
        .option("dataAddress", f"'{sheet}'!A2")
        .option("header", "true")
        .option("inferSchema", "false")
        .load(caminho)
    )

    # Verifica se todas as colunas esperadas pelo contrato
    # estão presentes na planilha.
    nomes = [n for n, _ in schema]
    faltando = [n for n in nomes if n not in raw.columns]
    assert not faltando, f"{arquivo}: sem colunas do contrato: {faltando}"

    # Seleciona apenas as colunas do contrato e realiza
    # a conversão para os tipos definidos.
    dados = [F.col(n).cast(t).alias(n) for n, t in schema]

    # Adiciona metadados de rastreabilidade da ingestão.
    df = (
        raw.select(
            *dados,
            F.input_file_name().alias("_source_file"),
            F.lit(INGESTION_TS).alias("_ingestion_timestamp")
        )
        .withColumnRenamed("ANO", "NU_ANO_AVALIACAO")
    )

    # Mantém apenas registros que representam dados válidos.
    # Linhas de rodapé, observações ou registros em branco
    # possuem o ano nulo após a conversão e são descartadas.
    df = df.filter(F.col("NU_ANO_AVALIACAO").isNotNull())

    # Grava os dados na camada Bronze em formato Parquet,
    # particionados pelo ano da avaliação.
    (
        df.write
        .mode("overwrite")
        .partitionBy("NU_ANO_AVALIACAO")
        .parquet(f"{BRONZE_BASE}/{entidade}")
    )

    return df

In [11]:
# ============================================================
# EXECUÇÃO DA INGESTÃO DAS PLANILHAS DE METAS
# ============================================================
#
# Executa a ingestão das planilhas de metas para a camada Bronze.
#
# Para cada combinação de entidade (Municípios ou UF) e ano,
# é utilizado o contrato correspondente, garantindo a leitura
# correta mesmo com as diferenças de schema entre as planilhas.
#
# São processadas as planilhas dos anos:
#   - 2023
#   - 2024
#   - 2025
#
# Saídas no S3:
#
#   s3a://{S3_BUCKET}/bronze/metas_municipios/
#   s3a://{S3_BUCKET}/bronze/metas_ufs/
#
# ============================================================
# EXECUÇÃO
# ============================================================

# ------------------------------------------------------------
# Metas 2023
# ------------------------------------------------------------
bronze_xlsx("metas_municipios", "resultados_e_metas_municipios.xlsx",
                          "Divulgação Alfabet Municipio", SCHEMA_METAS_MUN_2023, "2023")
bronze_xlsx("metas_ufs", "resultados_e_metas_ufs.xlsx",
                          "Divulgação Alfabet UF e Brasil", SCHEMA_METAS_UF_2023, "2023")
# ------------------------------------------------------------
# Metas 2024
# ------------------------------------------------------------
bronze_xlsx("metas_municipios", "resultados_e_metas_municipios_2024.xlsx",
            "Divulgação Alfabet Municipio", SCHEMA_METAS_MUN_2024, "2024")
bronze_xlsx("metas_ufs", "resultados_e_metas_ufs_2024_2.xlsx",
            "Divulgação Alfabet UF e Brasil", SCHEMA_METAS_UF_2024, "2024")

# ------------------------------------------------------------
# Metas 2025
# ------------------------------------------------------------
bronze_xlsx("metas_municipios", "resultados_e_metas_municipios_2025_v2.xlsx",
            "Divulgação Alfabet Municipio", SCHEMA_METAS_MUN_2025, "2025")
bronze_xlsx("metas_ufs", "resultados_e_metas_ufs_2025_v1.xlsx",
            "Divulgação Alfabet UF e Brasil", SCHEMA_METAS_UF_2025, "2025")

26/07/07 00:30:15 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
26/07/07 00:30:33 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/07/07 00:30:33 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/07/07 00:30:33 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/07/07 00:30:33 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/07/07 00:30:33 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/07/07 00:30:33 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,05

DataFrame[NU_ANO_AVALIACAO: int, CD_UF: string, SIGLA_UF: string, NOME_UF: string, REDE: string, PC_ALUNO_ALFABETIZADO_2023: double, PC_ALUNO_ALFABETIZADO_2024: double, PC_ALUNO_ALFABETIZADO_2025: double, META_FINAL_2024: string, META_FINAL_2025: string, META_FINAL_2026: string, META_FINAL_2027: string, META_FINAL_2028: string, META_FINAL_2029: string, META_FINAL_2030: string, PC_AVALIADOS_LP: double, _source_file: string, _ingestion_timestamp: timestamp]

### Validação da Ingestão da Camada Bronze

Após a ingestão das planilhas de metas, é realizada uma validação para confirmar que todos os arquivos foram gravados corretamente.

Nesta etapa verificamos:

- a quantidade de registros carregados;
- as partições criadas por ano;
- a união dos schemas das diferentes versões das planilhas;
- a preservação de registros importantes, como o agregado nacional ("Brasil").

Essas verificações garantem que a camada Bronze preservou corretamente os dados de origem antes das transformações realizadas na camada Silver.

In [12]:
# ============================================================
# VALIDAÇÃO DA INGESTÃO DAS PLANILHAS DE METAS
# ============================================================
#
# Verifica se as planilhas dos anos de 2023, 2024 e 2025 foram
# gravadas corretamente na camada Bronze.
#
# Durante a leitura é utilizado mergeSchema para unir os schemas
# dos diferentes anos, preservando as colunas específicas de
# cada versão das planilhas.
#
# São realizadas as seguintes verificações:
#   - quantidade de registros por entidade;
#   - anos presentes nas partições;
#   - quantidade de colunas após a união dos schemas;
#   - distribuição de registros por ano;
#   - preservação do registro "Brasil" nas planilhas de UF;
#   - schema final resultante da união dos anos.

# Lê todos os arquivos da entidade realizando a união dos
# schemas das diferentes versões das planilhas.
for ent in ["metas_municipios", "metas_ufs"]:
    
    # Lê todos os anos da entidade unificando os schemas.
    d = (
        spark.read
        .option("mergeSchema", "true")
        .parquet(f"{BRONZE_BASE}/{ent}")
    )
    # Obtém os anos disponíveis nas partições gravadas.
    anos = sorted(
        r[0]
        for r in d.select("NU_ANO_AVALIACAO").distinct().collect()
    )
    # Exibe um resumo da ingestão realizada.
    print(
        f"{ent}: {d.count():,} linhas | "
        f"partições {anos} | "
        f"{len(d.columns)} colunas"
    )

    # Confirma a distribuição de registros por ano.
    d.groupBy("NU_ANO_AVALIACAO") \
     .count() \
     .orderBy("NU_ANO_AVALIACAO") \
     .show()

# Verifica se o registro agregado do Brasil foi preservado
# durante a ingestão das planilhas de UF.
uf = (
    spark.read
    .option("mergeSchema", "true")
    .parquet(f"{BRONZE_BASE}/metas_ufs")
)

uf.filter(F.col("NOME_UF") == "Brasil") \
  .select(
      "NU_ANO_AVALIACAO",
      "NOME_UF",
      "REDE",
      "META_FINAL_2030",
      "PC_AVALIADOS_LP",
  ) \
  .orderBy("NU_ANO_AVALIACAO") \
  .show(truncate=False)

# Exibe o schema final após a união das versões de 2023,
# 2024 e 2025.
uf.printSchema()

metas_municipios: 16,286 linhas | partições [2023, 2024, 2025] | 22 colunas


+----------------+-----+
|NU_ANO_AVALIACAO|count|
+----------------+-----+
|            2023| 5468|
|            2024| 5352|
|            2025| 5466|
+----------------+-----+



metas_ufs: 84 linhas | partições [2023, 2024, 2025] | 21 colunas


+----------------+-----+
|NU_ANO_AVALIACAO|count|
+----------------+-----+
|            2023|   28|
|            2024|   28|
|            2025|   28|
+----------------+-----+



+----------------+-------+-------+---------------+---------------+
|NU_ANO_AVALIACAO|NOME_UF|REDE   |META_FINAL_2030|PC_AVALIADOS_LP|
+----------------+-------+-------+---------------+---------------+
|2023            |Brasil |PÚBLICA|> 80           |86.0           |
|2024            |Brasil |PÚBLICA|> 80           |87.37          |
|2025            |Brasil |PÚBLICA|> 80           |89.0           |
+----------------+-------+-------+---------------+---------------+

root
 |-- CD_UF: string (nullable = true)
 |-- SIGLA_UF: string (nullable = true)
 |-- NOME_UF: string (nullable = true)
 |-- REDE: string (nullable = true)
 |-- SAEB_2019: double (nullable = true)
 |-- SAEB_2021: double (nullable = true)
 |-- PC_ALUNO_ALFABETIZADO: double (nullable = true)
 |-- META_FINAL_2024: string (nullable = true)
 |-- META_FINAL_2025: string (nullable = true)
 |-- META_FINAL_2026: string (nullable = true)
 |-- META_FINAL_2027: string (nullable = true)
 |-- META_FINAL_2028: string (nullable = true)
 |-

In [13]:
# ============================================================
# VALIDAÇÃO DA INGESTÃO DAS PLANILHAS DE METAS
# ============================================================
#
# Lê os dados gravados na Bronze utilizando mergeSchema para
# unir os schemas dos diferentes anos e verifica:
#
#   - registros gravados por ano;
#   - partições criadas;
#   - quantidade de colunas após a união dos schemas.
#

for ent in ["metas_municipios", "metas_ufs"]:

    # Lê todos os anos da entidade unificando os schemas.
    d = (
        spark.read
        .option("mergeSchema", "true")
        .parquet(f"{BRONZE_BASE}/{ent}")
    )

    # Anos encontrados nas partições da Bronze.
    anos = sorted(
        r[0]
        for r in d.select("NU_ANO_AVALIACAO").distinct().collect()
    )

    # Distribuição de registros por ano.
    d.groupBy("NU_ANO_AVALIACAO") \
     .count() \
     .orderBy("NU_ANO_AVALIACAO") \
     .show()

    # Resumo da ingestão.
    print(
        f"{ent}: {d.count()} linhas | "
        f"partições {anos} | "
        f"{len(d.columns)} colunas na união\n"
    )

# Schema resultante da união dos anos para as metas de UF.
# É esperado que algumas colunas existam apenas em determinados
# anos (ex.: SAEB em 2023 e indicadores PC_* específicos por ano).
spark.read \
    .option("mergeSchema", "true") \
    .parquet(f"{BRONZE_BASE}/metas_ufs") \
    .printSchema()

+----------------+-----+
|NU_ANO_AVALIACAO|count|
+----------------+-----+
|            2023| 5468|
|            2024| 5352|
|            2025| 5466|
+----------------+-----+



metas_municipios: 16286 linhas | partições [2023, 2024, 2025] | 22 colunas na união



+----------------+-----+
|NU_ANO_AVALIACAO|count|
+----------------+-----+
|            2023|   28|
|            2024|   28|
|            2025|   28|
+----------------+-----+



metas_ufs: 84 linhas | partições [2023, 2024, 2025] | 21 colunas na união



root
 |-- CD_UF: string (nullable = true)
 |-- SIGLA_UF: string (nullable = true)
 |-- NOME_UF: string (nullable = true)
 |-- REDE: string (nullable = true)
 |-- SAEB_2019: double (nullable = true)
 |-- SAEB_2021: double (nullable = true)
 |-- PC_ALUNO_ALFABETIZADO: double (nullable = true)
 |-- META_FINAL_2024: string (nullable = true)
 |-- META_FINAL_2025: string (nullable = true)
 |-- META_FINAL_2026: string (nullable = true)
 |-- META_FINAL_2027: string (nullable = true)
 |-- META_FINAL_2028: string (nullable = true)
 |-- META_FINAL_2029: string (nullable = true)
 |-- META_FINAL_2030: string (nullable = true)
 |-- PC_AVALIADOS_LP: double (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- PC_ALUNO_ALFABETIZADO_2023: double (nullable = true)
 |-- PC_ALUNO_ALFABETIZADO_2024: double (nullable = true)
 |-- PC_ALUNO_ALFABETIZADO_2025: double (nullable = true)
 |-- NU_ANO_AVALIACAO: integer (nullable = true)

